In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from PyPDF2 import PdfReader
import gradio as gr

/Users/jabirkhan/Desktop/langchain-projects/agentic-ai-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(override=True)
openai = OpenAI()

In [4]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [5]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [7]:
push("HEYY!!")

Push: HEYY!!


In [8]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording name {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}
    

In [9]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [10]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters":{
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "the email address of the user"
            },
            "name": {
                "type": "string",
                "description": "the name of the user, if they provide it"
            },
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that is worth recording to give context"
            },
        },
        "required": ["email"],
        "additionalProperties": False
    },
}

In [11]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [13]:
tools = [{"type": "function", "function": record_user_details_json}, {"type": "function", "function": record_unknown_question_json}]

In [23]:



def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        print(f"tool called : {tool_name}", flush=True)
    # below could be written as:
    # result = globals()[tool_name](**arguments)
        result = ''
        if tool_name == "record_unknown_question_json":
            result = record_unknown_question_json(**tool_args)
        if tool_name == "record_user_details_json":
            result = record_user_details_json(**tool_args)
        
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results


In [16]:
reader = PdfReader("content/al-khawarizmi.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("content/text.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Al Khawarizmi"

In [17]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [18]:
import os

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [1]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    done = False
    while not done:
        response = gemini.chat.completions.create(model="gemini-3.5-flash", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        # if finish_reason is tool call we call the tool:
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results) # Adds each element from an iterable to the end of the list.
        else: 
            done = True
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch()